# Ejercicio 2 

Vamos a trabajar con un dataframe que contiene información sobre reseñas de videojuegos. Cada fila del dataframe contiene la información de una reseña, y las columnas son las siguientes:
- Contenido: texto de la reseña
- Valoración: Recomendado o No recomendado
- Recomendado binario: 1 si la valoración es Recomendado, 0 si la valoración es No recomendado
  
El objetivo de este ejercicio es analizar los comentarios con un modelo LLM, extrayendo la información relevante de los comentarios para poder hacer un análisis posterior.



# Análisis general del dataset de reseñas de videojuegos

In [9]:
import pandas as pd
from huggingface_hub import HfApi
from dotenv import load_dotenv
import os
import gc
gc.collect()

3907

In [11]:
from pydantic import BaseModel, Field, conlist
from typing import Literal, List
from lmformatenforcer import JsonSchemaParser

In [4]:
data = pd.read_csv('videogames_reviews.csv', index_col=0)
data.head(2)

,Contenido,Valoración,Recomendado_binario
0,2 marzo so bad,No recomendado,0
1,10 febrero actualmente recomiendo juego contab...,No recomendado,0


In [5]:
# Análisis exploratorio básico
print("Información básica del dataframe:")
print(f"Forma del dataframe: {data.shape}")
print(f"Columnas: {list(data.columns)}")
print("\nTipos de datos:")
print(data.dtypes)
print("\nPrimeras filas:")
data.head()

Información básica del dataframe:
Forma del dataframe: (20000, 3)
Columnas: ['Contenido', 'Valoración', 'Recomendado_binario']

Tipos de datos:
Contenido                str
Valoración               str
Recomendado_binario    int64
dtype: object

Primeras filas:


,Contenido,Valoración,Recomendado_binario
0,2 marzo so bad,No recomendado,0
1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,the world in this game is extremely static the...,No recomendado,0
4,zero replayability i finished this game in abo...,No recomendado,0


In [6]:
# Análisis de valores nulos y únicos
print("Valores nulos por columna:")
print(data.isnull().sum())
print("\nValores únicos en 'Valoración':")
print(data['Valoración'].value_counts())
print("\nEstadísticas de la columna 'Recomendado_binario':")
print(data['Recomendado_binario'].value_counts())

Valores nulos por columna:
Contenido              288
Valoración               0
Recomendado_binario      0
dtype: int64

Valores únicos en 'Valoración':
Valoración
No recomendado    10000
Recomendado       10000
Name: count, dtype: int64

Estadísticas de la columna 'Recomendado_binario':
Recomendado_binario
0    10000
1    10000
Name: count, dtype: int64


## Preprocesamiento simplificado

Objetivo: Obtener los 100 comentarios con más texto para análisis posterior.

In [125]:
# Preprocesamiento simplificado: Top 100 comentarios con más texto
print("Preprocesando datos para obtener los 100 comentarios más largos...")

# Eliminar filas con contenido nulo
df_limpio = data.dropna(subset=['Contenido']).copy()
print(f"Comentarios válidos (sin nulos): {len(df_limpio)}")

# Eliminar filas duplicadas
df_limpio = df_limpio.drop_duplicates(subset=['Contenido'])
print(f"Comentarios únicos después de eliminar duplicados: {len(df_limpio)}")

# Calcular longitud de caracteres
df_limpio['longitud_caracteres'] = df_limpio['Contenido'].str.len()

# Seleccionar los 100 comentarios más largos
df_top_100 = df_limpio.nlargest(100, 'longitud_caracteres').reset_index(drop=True)

# Eliminar columnas innecesarias
if 'Unnamed: 0' in df_top_100.columns:
    df_top_100 = df_top_100.drop('Unnamed: 0', axis=1)

print(f"\nDataset final: {len(df_top_100)} comentarios")
print(f"Longitud promedio: {df_top_100['longitud_caracteres'].mean():.1f} caracteres")
print(f"Longitud mínima: {df_top_100['longitud_caracteres'].min()} caracteres")
print(f"Longitud máxima: {df_top_100['longitud_caracteres'].max()} caracteres")

print("\nDistribución de valoraciones en el top 100:")
print(df_top_100['Valoración'].value_counts())

# Mostrar el dataframe resultante
df_top_100

Preprocesando datos para obtener los 100 comentarios más largos...
Comentarios válidos (sin nulos): 19712
Comentarios únicos después de eliminar duplicados: 14521

Dataset final: 100 comentarios
Longitud promedio: 6451.7 caracteres
Longitud mínima: 5121 caracteres
Longitud máxima: 7972 caracteres

Distribución de valoraciones en el top 100:
Valoración
No recomendado    63
Recomendado       37
Name: count, dtype: int64


,Contenido,Valoración,Recomendado_binario,longitud_caracteres
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972
1,this was probably my first preorder i felt that if the game turned out to be the worst game i played or didnt even run 10 dollars i paid for the witcher 2 and the witcher 3 felt so much like stealing that the extra 60 would still not cover the amount i would pay for the enjoyment i got from themafter putting 160 hours on cyberpunk 2077 i can say that it fully deserves the full price i paid for it on its own merits and after so many disappointments in the recent aaa releases that i had this is the highest praise that i can give to this game ill start by saying that i love urban spaces i love finding alleyways that i have never seen before in my real life neighbourhood i love seeing all those lit up windows at night and i love that there are so many details and slices of life in every corner in city city to is the greatest work the humanity can achieve due to its intricacy its organic nature and how it reflects the human condition since the liberty city of gta iv most games have been...,Recomendado,1,7662
2,oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty biased regarding cdpr and their games and thats why its painful to admit and accept the fact that in its current state cp 2077 is very dissapointing by cdpr standards and in general as wellim huge fantasy and rpg fan and with the witcher trilogy cdpr created one of my favorite gaming franchises i loved all 3 witcher games thronebreaker played gwent on gog for dozens of hours and all of them are entertaining quality content rich titles so my personal experiences with their previous products the countless hours i invested in them and the enjoyment i got out of them pretty much made cdpr my favorite developer studio and they even topped it with their attitude towards their community and the continous support they provide to their games patches and fixes years after release free dlcs paid expansions with incredible amount of contentas lot of other gamers i highly anticip...,No recomendado,0,7638
3,i know many will handwave away any criticisms of this game as salt or reply with git gud but as fan of fromsoft games since demons souls this game felt like the most punitive and least satisfying of the games while the setting is undoubtedly beautiful and there is fascinating story and interesting characters the design philosophy of this game is especially punishing to put it mildly this game feels as if the character the moveset of dark souls 3 with the additions of counter guards and jumping attacks while the enemies have the moveset of bloodborne foes and the tracking of sekiro enemies foes in this game are more punishing than ever with delayed attacks enhanced tracking and better ai while this works it feels like bit too much in some instances it feels like almost every enemy some 2880degree combo spin move giant area of effect explosion or some mix of b

# Selección de modelo para filtrado

Para la tarea de relevancia de una review, un LLM generalista permite aplicar criterios semánticos (mención de features, pros/contras, razones) y además extraer estructura en JSON en una segunda fase. 
La separación en dos prompts reduce errores: primero filtramos ruido; después extraemos atributos según se pide en el enuncaido de la entrega. 


In [8]:
# Primero cargamos token de huggingface desde .env
load_dotenv()
HF_API_KEY = os.getenv("HF_API_KEY")
if not HF_API_KEY:
    raise ValueError("La variable de entorno HF_API_KEY no está configurada. Por favor, configúrala en el archivo .env.")
api = HfApi(token=HF_API_KEY)

Un mismo modelo (misma arquitectura) puede tener varios checkpoints según el entrenamiento adicional que se le haya hecho. 

**Base model** — Modelo solo preentrenado para predecir el siguiente token; tiene conocimiento general pero no está optimizado para seguir instrucciones (tiende a continuar texto).

**Instruct model** — Base model fine-tuned con pares instrucción-respuesta para seguir instrucciones de forma directa (instruction tuning / a veces RLHF).

**Chat model** — Instruct model ajustado para conversaciones multi-turno, entrenado con diálogos user-assistant y diseñado para mantener contexto conversacional.


## Prueba del modelo local

Tras realizar las verificaciones necesarias y ejecutar el procesamiento en batch, la ejecución comenzó a alargarse más de lo debido. Revisando, vemos que el modelo se cargó con una ventana de contexto máxima (131072), lo que disparó memoria y tiempo: 

- CONTEXT 131072 (muy alto)
- SIZE 30 GB para llama3.1:8b

Por ello ajustamos "num_ctx": 2048 que define el tamaño de la ventana de contexto del modelo (cuántos tokens puede “tener en mente” a la vez entre prompt + historial + respuesta).

En práctica
- Más num_ctx -> más memoria usada y más lento, pero maneja textos más largos.
- Menos num_ctx -> menos memoria y más rápido, pero puede perder parte de contexto largo.
2048 suele ser un valor conservador para mejorar estabilidad/rendimiento cuando hay lentitud.


In [13]:
import requests
import json
import time

OLLAMA_URL = "http://localhost:11434/api/generate"

def ollama_generate(model: str, prompt: str, stream: bool = False, keep_alive: str = "30m", timeout_s: int = 180) -> str:
    """
    Llama a Ollama por HTTP y devuelve el texto generado.
    Incluye:
      - timeout (para que no se quede colgado)
      - manejo de errores y logs útiles
      - verificación de respuesta JSON
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "keep_alive": keep_alive, # mantiene el modelo en la RAM de Ollama para respuestas más rápidas en llamadas subsecuentes
        "options": {
            "temperature": 0.0,
            "num_ctx": 2048, # para modelos más grandes como llama3.1:8b, que tienen un contexto de 4096 tokens, es recomendable usar num_ctx=2048 o 4096 para evitar recortes prematuros del prompt
        }
    }

    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=timeout_s)
    except requests.exceptions.ConnectTimeout:
        raise RuntimeError("Timeout conectando a Ollama. ¿Está Ollama corriendo y escuchando en localhost:11434?")
    except requests.exceptions.ConnectionError:
        raise RuntimeError("No puedo conectar con Ollama en localhost:11434. Abre/arranca Ollama y prueba de nuevo.")
    except requests.exceptions.ReadTimeout:
        raise RuntimeError(f"Ollama tardó más de {timeout_s}s en responder. Sube timeout o usa un modelo más pequeño (llama3.2:3b).")

    # Si Ollama devuelve error HTTP, lo mostramos
    if not r.ok:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")

    # Intentar parsear JSON
    try:
        data = r.json()
    except json.JSONDecodeError:
        raise RuntimeError(f"Respuesta no-JSON de Ollama (primeros 500 chars):\n{r.text[:500]}")

    # Si hay error en el payload
    if "error" in data:
        raise RuntimeError(f"Ollama error: {data['error']}")

    # En modo stream=False, Ollama suele devolver {"response": "...", ...}
    if "response" not in data:
        raise RuntimeError(f"JSON inesperado devuelto por Ollama:\n{data}")

    return data["response"]


# ---------
# PRUEBA
# ---------
model = "llama3.1:8b"
prompt = "Explain what a videogame review is in 2 sentences."

t0 = time.time()
text = ollama_generate(model=model, prompt=prompt, stream=False, timeout_s=600)
print(f"⏱️ {time.time()-t0:.1f}s")
print(text)

⏱️ 8.1s
A videogame review is a written or video critique of a game that provides an evaluation of its quality, gameplay, and overall experience, often including opinions on its strengths and weaknesses. The review typically aims to inform potential players about the game's value and whether it is worth their time and money, providing a balanced assessment of the game's merits and flaws.



# Configuración de modelo para filtrado


Para este ejercicio se han considerado dos formas de ejecutar el modelo de lenguaje: utilizando un modelo local mediante Ollama y utilizando un modelo cargado directamente desde Hugging Face. Ambas opciones utilizan el mismo modelo base, pero se integran en el código de manera diferente.

*Opción A: uso de un modelo local a través de Ollama (primera iteración).*

- En esta primera fase se utiliza un modelo ya descargado y ejecutado en local mediante Ollama. Esto permite realizar pruebas rápidas sin necesidad de descargar pesos adicionales ni depender de una conexión externa.

- Ollama almacena los modelos en formatos optimizados para inferencia local basados en GGUF y ejecutados mediante el motor llama.cpp. Este motor está diseñado para ejecutar modelos de forma eficiente en CPU o GPU local mediante cuantización.

- Debido a esto, los modelos gestionados por Ollama no se cargan directamente con la librería Transformers de Hugging Face, ya que utilizan un formato interno diferente al de PyTorch. En lugar de cargar el modelo con Transformers, se interactúa con él mediante la API local de Ollama.

- Esta aproximación es especialmente útil para prototipado y experimentación rápida con modelos grandes sin necesidad de infraestructura adicional.

**Opción B: uso del modelo directamente desde Hugging Face (lo valoramos en segunda iteración).**   

- En la segunda iteración se plantea utilizar el mismo modelo pero cargado directamente desde el repositorio de Hugging Face. En este caso el modelo se descarga en formato compatible con PyTorch y puede utilizarse directamente con la librería Transformers.

- El modelo utilizado es Meta-Llama-3-8B-Instruct, que está optimizado para seguir instrucciones y realizar tareas de procesamiento de lenguaje natural como clasificación o extracción de información.

- Esta aproximación permite integrar el modelo directamente en el pipeline de Python, controlar parámetros de generación de forma más granular y utilizar herramientas del ecosistema de Hugging Face como tokenizers, pipelines o aceleración con GPU.
  
           

> En esta fase filtramos reseñas relevantes para extracción de insights de producto.  
> Una reseña puede ser relevante aunque su recomendación final sea negativa.  
> El criterio se basa en si aporta información concreta sobre aspectos del juego (gameplay, rendimiento, bugs, historia, etc.).


In [ ]:
from typing import List
from pydantic import BaseModel, Field
from string import Template
import json
import requests
import pandas as pd
import time
import re

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_FILTER = "llama3.1:8b"
TEMPERATURE_FILTER= 0.0
TOP_P_FILTER = 1.0
MAX_RESPONSE_TOKENS_FILTER = 1500
CONTEXT_WINDOW = 8192 # para modelos más grandes como llama3.1:8b, que tienen un contexto de 4096 tokens, es recomendable usar num_ctx=2048 o 4096 para evitar recortes prematuros del prompt

SYSTEM_PROMPT_STEP1 = """You are a videogame review analyst for product insights.
Return STRICT JSON only, with one root object: {"results":[...]}.
No markdown, no explanations, no extra keys, no extra text."""

USER_TEMPLATE_STEP1 = Template("""Classify each review as RELEVANT or IRRELEVANT for insight extraction. Reviews about any video game are in-scope.

Definition:
- RELEVANT: at least one concrete game aspect (e.g., gameplay, graphics, story, performance, bugs, AI, NPCs, customization, UI, controls, audio, world design) plus an explicit judgment about it.
- IRRELEVANT: no concrete aspects, generic praise/insult only, random/noisy text, repeated characters, or not about the game.

Decision policy:
- If a review has concrete aspects + evaluation, set relevant=true.
- Long or mixed reviews can still be relevant.
- Negative reviews can be relevant.
- Spanish and English reviews use the same criteria.
- If uncertain and no concrete aspect is present, set relevant=false.

Output rules:
- Return exactly one item per input review.
- Preserve IDs exactly as provided (R1, R2, ...).
- "relevant" must be boolean true/false.
- "reason" must be concise in English (max 12 words).
- Output ONLY one JSON object with key "results".

Few-shot examples:

Input:
<review id="R1">Great game, 10/10, amazing.</review>
Output:
{"results":[{"id":"R1","relevant":false,"reason":"Generic praise without concrete game aspects"}]}

Input:
<review id="R1">Yeeeeeeeeeeeeeeeeeeessss.</review>
Output:
{"results":[{"id":"R1","relevant":false,"reason":"Noisy repetitive text without meaningful game insight"}]}

Input:
<review id="R1">The city looks beautiful, but NPC AI is poor and repetitive.</review>
Output:
{"results":[{"id":"R1","relevant":true,"reason":"Concrete evaluation of visuals and NPC AI behavior"}]}

Input:
<review id="R1">Muchos bugs y bajones de FPS, pero historia y ambientación muy buenas.</review>
Output:
{"results":[{"id":"R1","relevant":true,"reason":"Specific judgments on performance, bugs, story, and atmosphere"}]}

REVIEWS:
$text
""")



def build_reviews_block(reviews: list[str]) -> str:
    lines = []
    for i, r in enumerate(reviews, 1):
        lines.append(f'<review id="R{i}">\n{(r or "").strip()}\n</review>')
    return "\n\n".join(lines)

def build_prompt_for_batch(reviews: list[str]) -> str:
    return f"{SYSTEM_PROMPT_STEP1}\n\n{USER_TEMPLATE_STEP1.substitute(text=build_reviews_block(reviews))}"

class ReviewItem(BaseModel):
    id: str = Field(description="Review ID, ex. R1")
    relevant: bool = Field(description="True only if the review mentions specific game aspects and a meaningful opinion; otherwise False.")
    reason: str = Field(description="Short reason, max 12 words")

class BatchOutput(BaseModel):
    results: List[ReviewItem]

def ollama_generate(*, prompt: str, timeout_s , max_response_tokens) -> str:
    payload = {
        "model": MODEL_FILTER,
        "prompt": prompt,
        "stream": False,
        "format": BatchOutput.model_json_schema(),      # fuerza salida JSON
        "keep_alive": "30m",
        "options": {
            "temperature": TEMPERATURE_FILTER,
            "top_p": TOP_P_FILTER,
            "num_predict": max_response_tokens,
            "num_ctx": CONTEXT_WINDOW,
        },
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=timeout_s)
    r.raise_for_status() # Si Ollama devuelve error HTTP, lanza excepción
    data = r.json() # Intenta parsear JSON

    if "error" in data: 
        raise RuntimeError(data["error"]) # Si existe data["error"] lanzamos un error 

    raw = (data.get("response") or "").strip()
    if not raw:
        raise RuntimeError(
            f"Empty response from Ollama. done_reason={data.get('done_reason')}, eval_count={data.get('eval_count')}"
        )
    return raw


def parse_json_robust(text: str) -> dict:
    text = (text or "").strip()
    if not text:
        raise ValueError("Empty response: no JSON to parse.")

    # 1) JSON directo
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2) JSON entre ```json ... ```
    if "```" in text:
        parts = text.split("```")
        for p in parts:
            p = p.replace("json", "", 1).strip()
            if p.startswith("{") and p.endswith("}"):
                return json.loads(p)

    # 3) recorte entre primera { y última }
    if "{" in text and "}" in text:
        cut = text[text.find("{"): text.rfind("}") + 1]
        return json.loads(cut)

    raise ValueError(f"No valid JSON found. Preview: {text[:300]!r}")

# Función para tolerar más esquemas alternativos en la respuesta JSON
def extract_items(data: dict) -> list[dict]:
    # Caso esperado data["results"]
    if isinstance(data.get("results"), list):
        return data["results"]

    # Caso alternativo frecuente data["review"]
    if isinstance(data.get("review"), dict):
        return [data["review"]]

    # Caso raíz como lista
    if isinstance(data, list):
        return data

    return []

# Prueba pare reordenar items por ID (R1, R2, ...)
# relevant y reason se contradicen: o el modelo decide mal o devuelve results en otro orden 
def _id_to_pos(x: str, n: int) -> int:
    # R1 -> 0, R2 -> 1...
    if isinstance(x, str) and x.startswith("R") and x[1:].isdigit():
        p = int(x[1:]) - 1
        if 0 <= p < n:
            return p
    return -1

def normalize_id(raw_id, fallback_idx: int) -> str:
    # fallback_idx es 1-based: 1 -> R1
    if raw_id is None:
        return f"R{fallback_idx}"

    s = str(raw_id).strip()

    # Ya viene bien: R1, r2...
    m = re.fullmatch(r"[Rr](\d+)", s)
    if m:
        return f"R{int(m.group(1))}"

    # Viene solo número: 1, "2"
    if s.isdigit():
        return f"R{int(s)}"

    # Cualquier otro caso raro
    return f"R{fallback_idx}"


def classify_batch(reviews, timeout_s , max_response_tokens):
    print(f"Generando prompt para batch de {len(reviews)} reseñas...", flush=True)
    prompt = build_prompt_for_batch(reviews)

    try:
        raw = ollama_generate(prompt=prompt, timeout_s=timeout_s, max_response_tokens=max_response_tokens)
        validated = BatchOutput.model_validate_json(raw)
        items = validated.results

        # n = len(reviews)

        # ordered = [None] * n
        # for r in items:
        #     pos = _id_to_pos(r.id, n)
        #     if pos == -1 or ordered[pos] is not None:
        #         raise ValueError(f"invalid_or_duplicate_id: {r.id}")
        #     ordered[pos] = {
        #         "id": r.id,
        #         "relevant": bool(r.relevant),
        #         "reason": str(r.reason).strip(),
        #     }

        # if any(x is None for x in ordered):
        #     raise ValueError("missing_ids_in_results")

        # return ordered

        if len(items) != len(reviews):
            raise ValueError(f"batch_mismatch: expected={len(reviews)} received={len(items)}")

        out = []
        for i, r in enumerate(items, start=1):
            out.append({
                "id": normalize_id(r.id, i),
                "relevant": bool(r.relevant),
                "reason": r.reason.strip(),
            })
        return out

    except Exception as e:
        print(f"[WARN] failure without retry: {e}", flush=True)
        return [
            {"id": f"R{i+1}", "relevant": False, "reason": "json_parse_error"}
            for i in range(len(reviews))
        ]


def run_step1_filter_df(
    df: pd.DataFrame,
    text_col: str = "Contenido",
    batch_size: int = 5,
    timeout_s: int = 600,
    max_response_tokens: int = MAX_RESPONSE_TOKENS_FILTER,
    sleep_s: float = 0.0
) -> pd.DataFrame:
    """
    Adds columns: relevant, reason
    Returns df with the new columns.
    """
    df = df.copy()
    texts = df[text_col].fillna("").astype(str).tolist()
    all_preds = []

    total = len(texts)
    total_batches = (total + batch_size - 1) // batch_size  # ceil division
  

    for batch_idx, start in enumerate(range(0, total, batch_size), start=1):
        end = min(start + batch_size, total)
        batch = texts[start:end]

        print(f"[Batch {batch_idx}/{total_batches}] rows {start+1}-{end} of {total}", flush=True)

        preds = classify_batch(
            batch,
            timeout_s=timeout_s,
            max_response_tokens=max_response_tokens
        )

        if len(preds) != len(batch):
            preds = [{
                "id": f"R{i+1}",
                "relevant": False,
                "reason": "batch_mismatch",
            } for i in range(len(batch))]

        all_preds.extend(preds)
        print(f"Processed: {len(all_preds)} / {total}", flush=True)

        if sleep_s > 0:
            time.sleep(sleep_s)

    df["relevant"] = [p["relevant"] for p in all_preds]
    df["reason"] = [p["reason"] for p in all_preds]
    return df



Para batch prompting, cuanto más corto, mejor. Si el prompt incluye ejemplos y bastante texto se le complica la tarea.

El tamaño del batch influye directamente en el número de tokens del prompt.
Cuantos más elementos se procesan en un batch, mayor es el número de tokens de entrada y mayor el tiempo de inferencia del modelo, por lo que puede ser necesario aumentar el timeout.

Por otro lado, al aumentar el tamaño del batch, el número de elementos devueltos en el JSON también aumenta, por lo que el número de tokens de salida debe ser suficiente para contener todos los resultados. Sin embargo, en tareas de clasificación el número de tokens de salida suele ser mucho menor que los tokens de entrada, ya que cada resultado ocupa pocas palabras. Por eso, el número máximo de tokens de salida debe mantenerse limitado para evitar respuestas demasiado largas y mejorar el rendimiento sin que ello provoque el truncamiento de la salida.

 Por este motivo, aunque en este caso nos nos afecte tanto al estar usando un modelo local, el principal factor que limita el tamaño del batch suele ser el número de tokens de entrada y la ventana de contexto del modelo.

#### **Primer intento:** 

Tenemos 100 reseñas procesadas. De las cuales: 
- 99 son relevantes
- 1 son irrelevantes
- 0 no se pudieron clasificar.

Como primera iteración los resultados no son, a simple vista, contradictorios ... solo 1 fila en el que la respuesta del modelo no sigue/corresponde con el tipado de la salida estructurada que le habíamos indicado en el prompt. Por el momento manejamos las respuestas que no siguen la estructura prevista con coerción en caso de error de lectura.  
Esto se debe a que el parámetro `format: "json"` obliga a que el modelo devuelva un JSON válido, pero no devuelve el esquema exacto que indicamos en el prompt. 

Por ello, definimos objetos de clase ReviewItem y BatchOutput para forzar la salida estructurada. Y ahora sí en el parámetro format especificamos el objeto exacto, tipado, que debe devolver en la respuesta. Sin embargo esto arroja algo de dificultad al ver salidas con relevant= True y iun reason incoherente que explica que el contenido es meaningfullness o generic... 
Esto nos obliga a afinar el prompt. Incluimos criterios generales, polícita de decision para decidir la relevancia y few-shots con incapié en los ejemplos negativos. 

Hemos subido la ventana de contexto a 4096 porque hay valores de la columna Contenido con mucho texto y seguimos observando inferencias incoherentes. 

In [55]:
df_scored = run_step1_filter_df(df_top_100[0:2], text_col="Contenido", batch_size=1, timeout_s=120, max_response_tokens=MAX_RESPONSE_TOKENS_FILTER, sleep_s=0.0)

[Batch 1/2] rows 1-1 of 2
Generando prompt para batch de 1 reseñas...
[WARN] fallo sin reintento: invalid_schema: {'review': 'This is a review of a product or service. It is a long string of text that appears to be a stream-of-consciousness review, with no clear structure or organization. The text is filled with 
Processed: 1 / 2
[Batch 2/2] rows 2-2 of 2
Generando prompt para batch de 1 reseñas...
Processed: 2 / 2


In [56]:
pd.set_option('display.max_colwidth', None)
df_scored[['relevant', 'reason']]

,relevant,reason
0,False,json_parse_error
1,True,"Mentions specific game aspects, gameplay, graphics, story, performance, bugs, difficulty, multiplayer, design, audio, and includes meaningful opinions"


#### **Segundo intento:**
  
Tenemos 100 reseñas procesadas. De las cuales: 
- 58 son relevantes
- 42 son irrelevantes
- 0 no se pudieron clasificar.

Analizamos las 42 reseñas falsas y vemos que el modelo es demasiado estricto y está castigando reviews largas como “generic”. No es problema de parseo (0 errores de clasificación técnica), es de `criterio semántico.`. 
Modificamos el prompt para quitar redundancias y enfatizar en las casuísticas vistas a en el conjunto de iteraciones realizadasd. 

In [18]:
# guardo el índice de filas irrelevantes para inspección posterior y para posibles iteraciones de mejora del prompt
irrelevant_index_df = pd.Index([ 0,  2,  6,  9, 13, 15, 20, 27, 28, 34, 35, 36, 37, 40, 41, 42, 46, 47,
       48, 54, 56, 57, 59, 71, 74, 76, 77, 78, 79, 80, 81, 82, 84, 86, 87, 88,
       90, 91, 93, 94, 96, 98],
      dtype='int64')

In [165]:
df_trial= run_step1_filter_df(df_top_100.loc[irrelevant_index_df].head(5), text_col="Contenido", batch_size=1, timeout_s=1200, max_response_tokens=MAX_RESPONSE_TOKENS_FILTER, sleep_s=0.0)

[Batch 1/10] rows 1-1 of 10
Generando prompt para batch de 1 reseñas...
Processed: 1 / 10
[Batch 2/10] rows 2-2 of 10
Generando prompt para batch de 1 reseñas...
Processed: 2 / 10
[Batch 3/10] rows 3-3 of 10
Generando prompt para batch de 1 reseñas...
Processed: 3 / 10
[Batch 4/10] rows 4-4 of 10
Generando prompt para batch de 1 reseñas...
Processed: 4 / 10
[Batch 5/10] rows 5-5 of 10
Generando prompt para batch de 1 reseñas...
Processed: 5 / 10
[Batch 6/10] rows 6-6 of 10
Generando prompt para batch de 1 reseñas...
Processed: 6 / 10
[Batch 7/10] rows 7-7 of 10
Generando prompt para batch de 1 reseñas...
Processed: 7 / 10
[Batch 8/10] rows 8-8 of 10
Generando prompt para batch de 1 reseñas...
Processed: 8 / 10
[Batch 9/10] rows 9-9 of 10
Generando prompt para batch de 1 reseñas...
Processed: 9 / 10
[Batch 10/10] rows 10-10 of 10
Generando prompt para batch de 1 reseñas...
Processed: 10 / 10


In [168]:
pd.set_option('display.max_colwidth', 200)
df_trial#[['relevant', 'reason']]

,Contenido,Valoración,Recomendado_binario,longitud_caracteres,relevant,reason
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972,False,Noisy repetitive text without meaningful game insight
2,oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty biased regarding cdpr and their games and thats why its painful to admit and ac...,No recomendado,0,7638,False,Long review with generic praise and criticism without concrete game aspects
6,in any industry this will always happen as an example like witcher that developed by the same company what they have shown and what they have given was different yet cuts covered only small portio...,No recomendado,0,7590,True,"Specific judgments on AI, NPCs, customization, UI, and game design"
9,28 enero cyberpunk 2077 labor of love or broken messi was not victim of the mass hype nor the outright lies that occurred prior to cyberpunk 2077s launch i knew nothing about the game postrelease ...,Recomendado,1,7583,True,"Specific judgments on gameplay, visuals, story, bugs, and immersion"
13,the short i didnt really think what everyone was clamoring for was the ubisoftification of dark souls the long prosjump buttonmusichorse combat mostlyfunaddictiveconsshockingly laggymovement is ja...,No recomendado,0,7536,True,"Specific judgments on various game aspects, including graphics, combat, and world design"
15,my review of cyberpunk 2077so ill start off by saying that i am huge fan of the witcher universe and especially the witcher 3 ive read the books and thoroughly played the game i was big fan of the...,No recomendado,0,7517,True,"Specific judgments on AI, character customization, replayability, and game features"
20,what i wish cyberpunk was truly openended rpg you can create anyone and be anyone you the player can set your own ambitions and the choices you make in the game would directly reflect who youre tr...,No recomendado,0,7416,True,"Specific judgments on game mechanics, visuals, and story pacing"
27,if youre looking for more quality reviews like this follow our curator page devils in the detailthe night city of reality the night city of expectationscyberpunk 2077 is game of grandiose ideals t...,Recomendado,1,7244,True,"Specific judgments on game mechanics, visuals, and story pacing"
28,producto recibido forma gratuita its now five years since skyrim first took over everyones lives with players happily spending hundreds of hours exploring what is one of the finest open world envi...,Recomendado,1,7150,True,"Concrete evaluation of visuals, performance, and game design"
34,here it is ladies and gentlemen our 8 years long waiting is over cyberpunk 2077 is finally here and it turned out to be disappoint pretty big one too well if youre console player you can actually ...,Recomendado,1,6963,True,"Specific judgments on gameplay, visuals, story, and game design"


Una vez validado el proceso ejecutamos para las 100 filas. 

In [169]:
df_scored = run_step1_filter_df(df_top_100, text_col="Contenido", batch_size=1, timeout_s=8000, max_response_tokens=MAX_RESPONSE_TOKENS_FILTER, sleep_s=0.0)

[Batch 1/100] rows 1-1 of 100
Generando prompt para batch de 1 reseñas...
Processed: 1 / 100
[Batch 2/100] rows 2-2 of 100
Generando prompt para batch de 1 reseñas...
Processed: 2 / 100
[Batch 3/100] rows 3-3 of 100
Generando prompt para batch de 1 reseñas...
Processed: 3 / 100
[Batch 4/100] rows 4-4 of 100
Generando prompt para batch de 1 reseñas...
Processed: 4 / 100
[Batch 5/100] rows 5-5 of 100
Generando prompt para batch de 1 reseñas...
Processed: 5 / 100
[Batch 6/100] rows 6-6 of 100
Generando prompt para batch de 1 reseñas...
Processed: 6 / 100
[Batch 7/100] rows 7-7 of 100
Generando prompt para batch de 1 reseñas...
Processed: 7 / 100
[Batch 8/100] rows 8-8 of 100
Generando prompt para batch de 1 reseñas...
Processed: 8 / 100
[Batch 9/100] rows 9-9 of 100
Generando prompt para batch de 1 reseñas...
Processed: 9 / 100
[Batch 10/100] rows 10-10 of 100
Generando prompt para batch de 1 reseñas...
Processed: 10 / 100
[Batch 11/100] rows 11-11 of 100
Generando prompt para batch de 1 

### Mostramos algunas observaciones

In [171]:
pd.set_option('display.max_colwidth', 200)
df_scored.sample(5)

,Contenido,Valoración,Recomendado_binario,longitud_caracteres,relevant,reason
55,good railroad story game not so great rpg there will be appropriately tagged spoilers for the ending because for the ending is easily where the game falls apart so i need to address it however don...,Recomendado,1,6295,True,"Specific judgments on game mechanics, story, and RPG elements"
79,actualización 4 borré juego instalé pc quedaba pegado haciendo formatié instalé driver anterior solucionó asco juego bugeadocon mas horas juego avanzando historia misiones secundarias mas habilida...,No recomendado,0,5421,True,"Specific judgments on game mechanics, visuals, and story pacing"
18,spoilersthis is one of those games where you wish steam had more than just yes or for reviews get ready for an excessive amount of commas because its time to dump some thoughtsmost of it boils dow...,No recomendado,0,7474,True,"Specific judgments on bugs, visuals, performance, and gameplay"
70,im so excited to be pointing the finger at this sport on steam everyone on the console feels really bad and the pc version worked very well for i was amazed at this game and spent more than 150 ho...,Recomendado,1,5789,True,"Specific judgments on game mechanics, visuals, and story"
54,first off i didnt watch any of the preview trailers i didnt know what was going on with cyberpunk i had assumed cdpr dumpstered it when i heard it droppedmy curiousity got the best of this is your...,No recomendado,0,6304,True,"Specific judgments on game mechanics, visuals, and story pacing"


In [ ]:
irrelevant_index_df = df_scored[df_scored["relevant"]==False][['Contenido', 'relevant', 'reason']].index

In [116]:
print(f"Tenemos {df_scored.shape[0]} reseñas procesadas.")

# Filtramos  dataframe con resultados 
df_relevant = df_scored[df_scored["relevant"]==True].copy()
df_irrelevant = df_scored[df_scored["relevant"]==False].copy()
df_others = df_scored[~df_scored["relevant"].isin([True, False])].copy()

print(f"De las cuales, {df_relevant.shape[0]} son relevantes, {df_irrelevant.shape[0]} son irrelevantes y {df_others.shape[0]} no se pudieron clasificar.")

Tenemos 100 reseñas procesadas.
De las cuales, 93 son relevantes, 7 son irrelevantes y 0 no se pudieron clasificar.


**Resultado final**

Tenemos 100 reseñas procesadas. De las cuales: 
- 93 son relevantes
- 7 son irrelevantes 
- 0 no se pudieron clasificar.

Revisamos las reseñas irrelevantes: 

In [117]:
df_irrelevant

,Contenido,Valoración,Recomendado_binario,longitud_caracteres,relevant,reason
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972,False,Noisy repetitive text without meaningful game insight
2,oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty biased regarding cdpr and their games and thats why its painful to admit and ac...,No recomendado,0,7638,False,Long review with generic praise and criticism without concrete game aspects
57,32 hours of gameplay and its finally over my character is riding shotgun in the scripted scene happy and free as tapeworm the game wants to put in the solemn and sentimental mood which is why it g...,No recomendado,0,6193,False,Generic praise and criticism without concrete game aspects
81,después haber arrastrado juego sesiones cortas largo periodo tiempo salida decidido hacer reseña reseña basa experiencia incompleta después casi 50 horas miseria sido incapaz arrastrar culo última...,No recomendado,0,5394,False,Noisy repetitive text without meaningful game insight
86,aviso criticar bugs rendimiento entendible juego recién salidopuntos positivosgráficos sinceramente ve increíble absolutamente cualquier aspecto grafico genial reflejos noches lluvia increíbles gr...,No recomendado,0,5343,False,Generic praise without concrete game aspects
93,16 mayo verdad visto muchas crãticas juego diciendo horrible nefasto solo gustarã si fan harry potter parte razã³nla verdad hogwarts legacy juego lanzado solo contentar agradar fieles fans porquã ...,Recomendado,1,5263,False,Generic praise without concrete game aspects
94,16 marzo suelo hacer análisis menos aun steam dado avances previos pintaba mejores juegos año tremenda decepción sido conforme mas jugaba apetecía expresar opiniones quedado ambientación increíble...,No recomendado,0,5243,False,Noisy repetitive text without meaningful game insight


De las filas clasificadas como irrelevantes, observamos que varias son **casos frontera**: reseñas largas, con redacción poco limpia o mezcla de contenido, que el modelo interpreta como poco informativas aunque sí contienen señales útiles.

Esto sugiere que el filtro podría afinarse más reforzando el prompt en tres puntos:

- **Diferenciar ruido real vs texto largo desestructurado**  
  Indicar explícitamente que una reseña larga puede seguir siendo relevante si menciona aspectos concretos y valoración.

- **Mejorar criterio para reseñas en español y texto con errores de codificación**  
  Añadir instrucciones y few-shots específicos en español para evitar falsos negativos por calidad lingüística.

- **Reforzar casos de crítica mixta**  
  Especificar que una reseña con pros y contras sigue siendo relevante cuando aporta información accionable sobre gameplay, rendimiento, bugs, historia, etc.

No obstante, dado que el número de filas afectadas es reducido y no desvirtúa el resultado global, no se dedicó más iteración en esta fase.


# Guardamos resultados

In [ ]:
os.makedirs("data/processed", exist_ok=True)
df_scored.to_csv("data/processed/reviews_filtered.csv")
df_scored.to_pickle("data/processed/reviews_filtered.pkl")

In [ ]:
#df_scored = pd.read_pickle("data/processed/reviews_filtered.pkl") 
#df_scored = pd.read_csv("data/processed/reviews_filtered.csv", index_col=0)

# Feature extraction

Desde el punto de vista de **gestión de producto en videojuegos**, estas features permiten transformar las reseñas en **información estructurada** que facilite la identificación de insights relevantes para el equipo de desarrollo.

- **sentiment:** positive | negative | mixed | neutral  
  Indica la **orientación general de la opinión** expresada en la reseña.

- **sentiment_score:** puntuación de la **intensidad global de la opinión** en una escala de **0–10**, independientemente de si la review es positiva o negativa.  
  - **0–3** → sentimiento negativo  
  - **4–6** → sentimiento mixto o neutral  
  - **7–10** → sentimiento positivo

- **aspects:** lista de **aspectos del juego mencionados en la reseña** (máximo **5 términos**).  
  Ejemplos de aspectos posibles: gameplay, graphics, story, performance, difficulty, multiplayer, controls, audio, world design.  
  Esto permite realizar análisis como el porcentaje de reseñas que mencionan **problemas de rendimiento** o la frecuencia de menciones a **gameplay** o **multiplayer**.

- **issue_type:** identifica si la reseña menciona algún **problema técnico o bug**.  
  Ejemplos de valores posibles: bug, performance_issue, balance_issue, technical_problem, none.  
  Esta feature permite detectar rápidamente **problemas recurrentes en el juego**.

- **feedback_type:** clasificación del tipo de feedback recibido.  
  - praise → elogio o comentario positivo  
  - complaint → queja o crítica  
  - mixed → combinación de elogios y críticas

  Esta variable es muy utilizada en **product analytics**, ya que permite distinguir entre comentarios que destacan **fortalezas del producto** y aquellos que señalan **problemas o áreas de mejora**.



## Selección de modelo para feature extraction

Usamos el modelo `Qwen/Qwen2.5-7B-Instruct, modelo instruct, de 7B de parámetros. (similar al de Llama que hemos empleado con Ollama) pero en este caso lo descargamos de Hugging Face 

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLMfrom transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" 
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device in {"mps", "cuda"} else torch.float32 # dtype: en mi caso con MAC M1 o uso MPS con precisión float16; o uso CPU con float32

## Cargamos tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
# Algunos modelos no traen pad_token definido
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Cargamos modelo 
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map=None).to(device)
model.eval()
print("Modelo cargado .")

Dispositivo: mps
Cargando tokenizer...
Cargando modelo...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modelo cargado correctamente.


### Qwen/Qwen2.5-7B-Instruct en local

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc, torch
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

In [ ]:
import torch, time

model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

@torch.no_grad()
def hf_hello():
    messages = [
        {"role": "system", "content": "You are helpful. Reply with exactly: hola"},
        {"role": "user", "content": "Say hola"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    t0 = time.time()
    out = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        temperature= None, 
        top_p= None,
        top_k= None,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    dt = time.time() - t0

    gen = out[0][inputs["input_ids"].shape[1]:]
    txt = tokenizer.decode(gen, skip_special_tokens=True).strip()
    print(f"Respuesta: {txt!r}")
    print(f"Tiempo: {dt:.1f}s")

hf_hello()



/opt/miniconda3/envs/gf311_cf_entrega/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/miniconda3/envs/gf311_cf_entrega/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/miniconda3/envs/gf311_cf_entrega/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Respuesta: 'holaa'
Tiempo: 176.8s


#### <span style="color:red;">Con cpu y este ejemplo (que es un simple "Say hello") el modelo tarda 177 segundos (3 minutos aprox) en generar la respuesta! (DEMASIADO)</span>

--- 

#### ¿Por qué Ollama 8B puede ir más rápido que Qwen 7B?

Ollama usa llama.cpp muy optimizado (Apple Silicon/MPS/Metal) y normalmente corre modelos cuantizados (4-bit/5-bit), mucho más ligeros.  
En HF cargué el modelo “completo” (más pesado en memoria y ancho de banda). 

--- 

#### ¿Tiene sentido coger un modelo grande pero cuantizado? 

Un modelo grande cuantizado suele dar mejor calidad que uno pequeño no cuantizado, con coste razonable.
  
Trade-off:  
  
- 7B quantized (4/5-bit): mejor calidad semántica, menos memoria/latencia que 7B full
- 1.5B full: más rápido, peor calidad en tareas finas/consistencia

--- 

#### Palancas importantes 

Qué podemos controlar mejor en Hugging Face. Hay 5 palancas importantes:

- **1. El dtype**
No cargar el modelo en precisión alta si no hace falta.

- **2. use_cache**
La KV cache acelera, pero también consume memoria.

- **3. max_new_tokens**
Cuantos más tokens genere, más crece la cache.

- **4. batch_size**
Más entradas a la vez = más memoria.

- **5. El propio backend** En Mac, Transformers + MPS no siempre es la opción más eficiente. Para Apple Silicon suelen ir mejor:  
  
•	Ollama + llama.cpp.  
•	MLX / mlx-lm (sirve como motor de inferencia optimizado para Apple Silicon). 
      
En cuyo caso, Hugging Face nos sirve como hub de modelos. 

--- 

#### Opciones


Nuestra idea era emplear Torch para usar las funciones vistas en clase: 

- `AutoTokenizer.from_pretrained(...)`
  
- `AutoModelForCausalLM.from_pretrained(...)`

En Hugging Face podemos encontrar modelos cuantizados en varios formatos:

- **GGUF**  
  Pensado para llama.cpp, Ollama, LM Studio, etc.  
  Probado en la fase 1 de filtrado: estable y práctico en Mac.

- **bitsandbytes (4-bit / 8-bit)**  
  Implica cuantización al vuelo (no siempre descargas un repo ya cuantizado).  
  Cargas el modelo base y aplicas cuantización con BitsAndBytesConfig.  
  Está más orientado a CUDA/NVIDIA.

- **AWQ / GPTQ**  
  Transformers soporta AWQ y GPTQ con matices según plataforma.  
  En Apple Silicon, estas combinaciones no suelen estar bien soportadas en transformers estándar.

- **Metal / Apple Silicon quantization**  
  Transformers también documenta MetalConfig con opciones específicas para MPS/Apple Silicon.  
  En la práctica, muchas combinaciones siguen limitadas.  
  Que exista esa documentación no implica que cualquier modelo cuantizado (por ejemplo GPTQ/AWQ) funcione en MPS.

   

--- 
> **Nota:** Priorizamos una ejecución local con Hugging Face por su valor didáctico y por permitir mayor control sobre la inferencia. No obstante, dado que la ejecución local de modelos cuantizados en Apple Silicon puede presentar limitaciones de memoria y compatibilidad, vamos a emplear como alternativa el uso de una API para garantizar la estabilidad del pipeline y completar la extracción estructurada.
---



In [25]:
import os 
from dotenv import load_dotenv 
from groq import Groq

# Cargamos tojen de autenticación desde archivo de variables de entorno 
load_dotenv() 

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("La variable de entorno GROQ_API_KEY no está configurada. Por favor, configúrala en el archivo .env.")   
print("Token cargado correctamente.")

client = Groq(api_key=GROQ_API_KEY)

Token cargado correctamente.


### Análisis de rate limits de los modelos que aceptan output estructurado

#### Trade off en cuanto a la selección de modelo

- **GPT-OSS 20B / 120B**  
    Ventaja: strict: true real.Son los únicos modelos que garantizan *schema-compliant output*.  
    Desventaja: en free tier tienen límites más modestos con <span style="color:green;">30 RPM, 8K TPM Y 200K TPD</span>.  ￼
  
- **Kimi K2 Instruct 0905**   
    Ventaja: más margen de rate limits en free tier: <span style="color:green;">60 RPM, 10K TPM Y 300K</span>.  
    Desventaja: admite salida estructurada pero con *strict: false* (best-effort, puede dar error) no constrained decoding.  ￼
  
- **Llama 4 Scout**   
    Ventaja: bastante más TPM en free tier: <span style="color:green;">30 RPM, 30K TPM Y 500K TPD</span>.   
    Desventaja: admite salida estructurada pero con *strict: false* (best-effort, puede dar error) no constrained decoding. 

### openai/gpt-oss-20b, batch_size= 2, y sleep_s = 60 

Con esta configuración, la ejecución fue estable porque reduce la presión sobre los límites de la API y mantiene el tamaño de cada respuesta dentro de un rango controlable.

Según aumentamos el batch, aumenta el prompt/tokens de entrada y el JSON/tokens de salida, es decir, sube la carga total de la llamada y es más probable que superemos el ratio de **max_tokens** o el **TPM (tokens por minuto) de la API** especialmente en free tier. 

En la función de groq_generate_from_messages usamos el parámetro **strict: true** compatible con los modelos openai/gpt-oss-20b y openai/gpt-oss-120b que obligan a la salida estructurada y garantiza el ajuste exacto al schema Pydantic definido para la salida. Además, groq obliga a definir el el schema que todos los campos sean obligatorios. 

#### Construcción del pipeline

In [ ]:
# Parámetros de inferencia 
MODEL_ID = "openai/gpt-oss-20b"
MAX_RESPONSE_TOKENS_FEAT = 700
TEMPERATURE_FEAT = 0 # clasificación => determinista
TOP_P_FEAT = 1.0
MAX_RETRIES = 2 

# Si el modelo soporta structured outputs con json_schema -->  True.
# https://console.groq.com/docs/structured-outputs?utm_source=chatgpt.com#supported-models 
# Sino ponerlo a False y se validará la salida con prompt + Pydantic.

In [ ]:
from string import Template
import re
import json
import time
import pandas as pd
from groq import Groq
from pydantic import BaseModel, Field, conlist, ConfigDict
from typing import Literal, List

#Prompt
SYSTEM_PROMPT_STEP2 = """
You are a videogame review analyst for a product development team.
Extract structured product insights from each review.
"""

USER_TEMPLATE_STEP2 = Template("""
For each review, extract:
- sentiment
- sentiment_score
- aspects
- feedback_type

Rules:
- Reviews can be in English or Spanish; apply the same criteria.
- Return exactly one result per input review.
- Preserve IDs exactly as provided (R1, R2, ...).
- Do not invent information not present in the review.
- Aspects must describe general game features (e.g., gameplay, graphics, story, AI, performance, world design, combat, audio, bugs, customization).
- Avoid very specific entities such as location names, characters, or lore elements.
- Only include aspects explicitly mentioned in the review.
- Use at most 5 aspects. Never return more than 5 items.
- Output ONLY one JSON object with key "results".

REVIEWS:
$text
""")


# Pydantic schemas para validación de respuesta
# Groq exige que todos los objetos del JSON Schema tengan additionalProperties: false.
class ReviewFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")
    id: str
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    sentiment_score: int = Field(ge=0, le=10)
    aspects: conlist(str, max_length=5)
    feedback_type: Literal["praise", "complaint", "mixed"]

class BatchOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    results: List[ReviewFeatures]

schema = BatchOutput.model_json_schema()   # Pydantic v2



# Helpers
def build_reviews_block_step2(reviews: list[str]) -> str:
    lines = []
    for i, r in enumerate(reviews, 1):
        rid = f"R{i}"
        r = (r or "").strip()
        lines.append(f'<review id="{rid}">\n{r}\n</review>')
    return "\n\n".join(lines)

def build_messages_for_batch_step2(reviews: list[str]) -> list[dict]:
    reviews_block = build_reviews_block_step2(reviews)
    user_prompt = USER_TEMPLATE_STEP2.substitute(text=reviews_block)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_STEP2},
        {"role": "user", "content": user_prompt},
    ]
    return messages



# Función para llamar a Groq
def groq_generate_from_messages(messages: list[dict], max_tokens: int = MAX_RESPONSE_TOKENS_FEAT) -> str:
    print("MODEL_ID usado:", MODEL_ID)
    completion = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=TEMPERATURE_FEAT,
        max_tokens=max_tokens,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "batch_output",
                "strict": True,
                "schema": schema
            }
        }
    )

    return completion.choices[0].message.content



# Extraccion en batches 

def extract_features_batch_groq(reviews: list[str], max_retries: int = MAX_RETRIES) -> list[dict]:
    print(f"\n--- Extrayendo features de batch de {len(reviews)} reseñas ---", flush=True)

    messages = build_messages_for_batch_step2(reviews)

    for attempt in range(1, max_retries + 1):
        try:
            start = time.time()
            raw = groq_generate_from_messages(messages)
            elapsed = time.time() - start

            print(f"Groq respondió en {elapsed:.1f} segundos", flush=True)
            print("Primeros 200 caracteres de la respuesta:", flush=True)
            print(raw[:200], flush=True)

            data = json.loads(raw)
            validated = BatchOutput(**data) # Mantenemos validación Pydantic 
            results = validated.results

            if len(results) != len(reviews):
                raise ValueError(f"batch_mismatch: expected={len(reviews)} got={len(results)}")

            out = []
            for i, r in enumerate(results, start=1):
                out.append({
                    "id": r.id.strip() if r.id else f"R{i}",
                    "sentiment": r.sentiment,
                    "sentiment_score": r.sentiment_score,
                    "aspects": [a.strip().lower() for a in r.aspects][:5],
                    "feedback_type": r.feedback_type,
                })

            print("Batch procesado\n", flush=True)
            return out

        except Exception as e:
            print(f"[WARN] intento {attempt}/{max_retries} fallido en step2: {e}", flush=True)

            if attempt == max_retries:
                return [{
                    "id": f"R{i+1}",
                    "sentiment": "neutral",
                    "sentiment_score": 5,
                    "aspects": [],
                    "feedback_type": "mixed"
                } for i in range(len(reviews))]
            

# Workflow 
def run_step2_feature_extraction_groq( df: pd.DataFrame, text_col: str = "Contenido", batch_size: int = 1, sleep_s: float = 0.0) -> pd.DataFrame:
    df = df.copy()
    texts = df[text_col].fillna("").astype(str).tolist()
    all_preds = []

    total = len(texts)
    total_batches = (total + batch_size - 1) // batch_size

    for batch_idx, start in enumerate(range(0, total, batch_size), start=1):
        end = min(start + batch_size, total)
        batch = texts[start:end]

        print(f"[Batch {batch_idx}/{total_batches}] rows {start+1}-{end} of {total}", flush=True)
        preds = extract_features_batch_groq(batch)

        if len(preds) != len(batch):
            preds = [{
                "id": f"R{i+1}",
                "sentiment": "neutral",
                "sentiment_score": 5,
                "aspects": [],
                "feedback_type": "mixed"
            } for i in range(len(batch))]

        all_preds.extend(preds)

        if sleep_s > 0:
            time.sleep(sleep_s)

    df["sentiment"] = [p["sentiment"] for p in all_preds]
    df["sentiment_score"] = [p["sentiment_score"] for p in all_preds]
    df["aspects"] = [p["aspects"] for p in all_preds]
    df["feedback_type"] = [p["feedback_type"] for p in all_preds]

    return df

#### Pruebas unitarias

In [93]:
reviews_test = df_to_evaluate["Contenido"].fillna("").astype(str).tolist()[:1]
messages_test = build_messages_for_batch_step2(reviews_test)
print(messages_test[1]["content"])


For each review, extract:
- sentiment
- sentiment_score
- aspects
- feedback_type

Rules:
- Reviews can be in English or Spanish; apply the same criteria.
- Return exactly one result per input review.
- Preserve IDs exactly as provided (R1, R2, ...).
- Do not invent information not present in the review.
- Aspects must describe general game features (e.g., gameplay, graphics, story, AI, performance, world design, combat, audio, bugs, customization).
- Avoid very specific entities such as location names, characters, or lore elements.
- Only include aspects explicitly mentioned in the review.
- Use at most 5 aspects. Never return more than 5 items.
- Output ONLY one JSON object with key "results".

REVIEWS:
<review id="R1">
this was probably my first preorder i felt that if the game turned out to be the worst game i played or didnt even run 10 dollars i paid for the witcher 2 and the witcher 3 felt so much like stealing that the extra 60 would still not cover the amount i would pay for 

In [ ]:
preds_test = extract_features_batch_groq(reviews_test)


--- Extrayendo features de batch de 1 reseñas ---


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 0.6 segundos
Primeros 200 caracteres de la respuesta:
{"results":[{"id":"R1","sentiment":"positive","sentiment_score":9,"aspects":["city design","verticality","quest variety","difficulty level","story depth"],"feedback_type":"praise"}]}
Batch procesado



[{'id': 'R1',
  'sentiment': 'positive',
  'sentiment_score': 9,
  'aspects': ['city design',
   'verticality',
   'quest variety',
   'difficulty level',
   'story depth'],
  'feedback_type': 'praise'}]

In [ ]:
df_test = run_step2_feature_extraction_groq(df_to_evaluate[0:10], text_col="Contenido", batch_size=2, sleep_s= 60)
df_test

[Batch 1/5] rows 1-2 of 10

--- Extrayendo features de batch de 2 reseñas ---


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 1.0 segundos
Primeros 200 caracteres de la respuesta:
{"results":[{"id":"R1","sentiment":"positive","sentiment_score":9,"aspects":["city design","verticality","detail level","character variety","story depth"],"feedback_type":"praise"},{"id":"R2","sentime
Batch procesado

[Batch 2/5] rows 3-4 of 10

--- Extrayendo features de batch de 2 reseñas ---


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 1.5 segundos
Primeros 200 caracteres de la respuesta:
{"results":[{"id":"R1","sentiment":"negative","sentiment_score":2,"aspects":["combat","story","graphics","performance","gameplay"],"feedback_type":"complaint"},{"id":"R2","sentiment":"neutral","sentim
Batch procesado

[Batch 3/5] rows 5-6 of 10

--- Extrayendo features de batch de 2 reseñas ---


...

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 1.0 segundos
Primeros 200 caracteres de la respuesta:
{"results":[{"id":"R1","sentiment":"mixed","sentiment_score":5,"aspects":["performance","bugs","story","characters","open world"],"feedback_type":"mixed"},{"id":"R2","sentiment":"mixed","sentiment_sco
Batch procesado

[Batch 5/5] rows 9-10 of 10

--- Extrayendo features de batch de 2 reseñas ---


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 0.9 segundos
Primeros 200 caracteres de la respuesta:
{"results":[{"id":"R1","sentiment":"negative","sentiment_score":2,"aspects":["driving","AI","combat","story","bugs"],"feedback_type":"complaint"},{"id":"R2","sentiment":"mixed","sentiment_score":5,"as
Batch procesado



,Contenido,Valoración,Recomendado_binario,longitud_caracteres,relevant,reason,sentiment,sentiment_score,aspects,feedback_type
1,this was probably my first preorder i felt tha...,Recomendado,1,7662,True,"Concrete evaluation of visuals, gameplay, stor...",positive,9,"[city design, verticality, detail level, chara...",praise
3,i know many will handwave away any criticisms ...,No recomendado,0,7609,True,"Concrete evaluation of game mechanics, visuals...",negative,3,"[difficulty, reward system, boss design, world...",complaint
4,i had enormous expectations having put thousan...,No recomendado,0,7609,True,"Specific judgments on game mechanics, story, a...",negative,2,"[combat, story, graphics, performance, gameplay]",complaint
5,i finished the witcher 3 for the first time th...,Recomendado,1,7604,True,"Concrete evaluation of story, characters, worl...",neutral,6,"[story, characters, world, bugs, gameplay]",mixed
6,in any industry this will always happen as an ...,No recomendado,0,7590,True,"Specific judgments on AI, NPCs, customization,...",negative,2,"[ai behavior, character customization, gamepla...",complaint
7,now my favorite fromsoft game tldr here phenom...,Recomendado,1,7588,True,"Specific judgments on combat, world, story, pe...",positive,8,"[combat, world design, performance, story, bos...",praise
8,i know im quite late to the review party and e...,Recomendado,1,7587,True,"Specific judgments on visuals, NPC AI, bugs, a...",mixed,5,"[performance, bugs, story, characters, open wo...",mixed
9,28 enero cyberpunk 2077 labor of love or broke...,Recomendado,1,7583,True,"Specific judgments on gameplay, visuals, story...",mixed,4,"[bugs, open world, story, characters, combat]",mixed
10,ive been playing this game to death the past f...,No recomendado,0,7578,True,"Specific judgments on various game aspects, in...",negative,2,"[driving, ai, combat, story, bugs]",complaint
11,new generation of openworld adventure this gam...,Recomendado,1,7571,True,"Specific judgments on graphics, world design, ...",mixed,5,"[graphics, story, character creation, ai, loot...",mixed


### meta-llama/llama-4-scout-17b-16e-instruct, batch_size= 2, y sleep_s = 60 

Empleamos el modelo de meta ya que es el que mayor ratio de Tokens Per Day admite, y es compatible con la salida estructurada aunque en modo *best-effort*, no de forma estricta. 

Con el modelo seleccionado y la configuración establecida, podemos cubrir hasta 2 iteraciones completas del dataframe (93 filas) por día sin alcanzar el límite de tokens por día (TPD), que en este proyecto es el principal factor restrictivo.
 
Incorporamos funciones de para parsear el json de salida y para normalizar el identificador de las reviews que eliminamos en con el modelo de OpenAI ya que al ofrecer Strict Mode no vimos necesario. 

Inocrporamos dichas funciones al workflow y realizamos las pruebas de validación. 

In [94]:

def parse_json_robust(text: str) -> dict:
    text = (text or "").strip()
    if not text:
        raise ValueError("Empty response: no JSON to parse.")

    # 1) JSON directo
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2) JSON entre ```json ... ```
    if "```" in text:
        parts = text.split("```")
        for p in parts:
            p = p.replace("json", "", 1).strip()
            if p.startswith("{") and p.endswith("}"):
                return json.loads(p)

    # 3) recorte entre primera { y última }
    if "{" in text and "}" in text:
        cut = text[text.find("{"): text.rfind("}") + 1]
        return json.loads(cut)

    raise ValueError(f"No valid JSON found. Preview: {text[:300]!r}")

def normalize_id(raw_id, fallback_idx: int) -> str:
    # fallback_idx es 1-based: 1 -> R1
    if raw_id is None:
        return f"R{fallback_idx}"

    s = str(raw_id).strip()

    # Ya viene bien: R1, r2...
    m = re.fullmatch(r"[Rr](\d+)", s)
    if m:
        return f"R{int(m.group(1))}"

    # Viene solo número: 1, "2"
    if s.isdigit():
        return f"R{int(s)}"

    # Cualquier otro caso raro
    return f"R{fallback_idx}"

# Extraccion en batches 
def extract_features_batch_groq(reviews: list[str], max_retries: int = MAX_RETRIES) -> list[dict]:
    print(f"\n--- Extrayendo features de batch de {len(reviews)} reseñas ---", flush=True)

    messages = build_messages_for_batch_step2(reviews)

    for attempt in range(1, max_retries + 1):
        try:
            start = time.time()
            raw = groq_generate_from_messages(messages)
            elapsed = time.time() - start

            print(f"Groq respondió en {elapsed:.1f} segundos", flush=True)

            data = parse_json_robust(raw)
            items = data.get("results", [])
            validated = BatchOutput(results=items)
            results = validated.results

            if len(results) != len(reviews):
                raise ValueError(f"batch_mismatch: expected={len(reviews)} got={len(results)}")

            out = []
            for i, r in enumerate(results, start=1):
                out.append({
                    "id": normalize_id(r.id, i),
                    "sentiment": r.sentiment,
                    "sentiment_score": r.sentiment_score,
                    "aspects": [a.strip().lower() for a in r.aspects][:5],
                    "feedback_type": r.feedback_type,
                })

            print("Batch procesado\n", flush=True)
            return out

        except Exception as e:
            print(f"[WARN] intento {attempt}/{max_retries} fallido en step2: {e}", flush=True)

            if attempt == max_retries:
                return [{
                    "id": f"R{i+1}",
                    "sentiment": "neutral",
                    "sentiment_score": 5,
                    "aspects": [],
                    "feedback_type": "mixed"
                } for i in range(len(reviews))]



#### Pruebas unitarias

In [95]:
# Parámetros de inferencia 
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"
MAX_RESPONSE_TOKENS_FEAT = 700
TEMPERATURE_FEAT = 0 # clasificación => determinista
TOP_P_FEAT = 1.0
MAX_RETRIES = 3 

In [78]:
df_test = run_step2_feature_extraction_groq(df_to_evaluate[0:8], text_col="Contenido", batch_size=2, sleep_s= 30)
df_test

[Batch 1/4] rows 1-2 of 8

--- Extrayendo features de batch de 2 reseñas ---
MODEL_ID usado: meta-llama/llama-4-scout-17b-16e-instruct


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 1.5 segundos
Primeros 200 caracteres de la respuesta:
{
"results": [
    {
      "id": "R1",
      "sentiment": "positive",
      "sentiment_score": 9,
      "aspects": [
        "city design",
        "gameplay freedom",
        "story",
        "charac
Batch procesado

[Batch 2/4] rows 3-4 of 8

--- Extrayendo features de batch de 2 reseñas ---
MODEL_ID usado: meta-llama/llama-4-scout-17b-16e-instruct


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 0.7 segundos
Primeros 200 caracteres de la respuesta:
{
"results": [
    {
      "id": "R1",
      "sentiment": "negative",
      "sentiment_score": 2,
      "aspects": [
        "game design",
        "story",
        "combat",
        "multiplayer",
  
Batch procesado

[Batch 3/4] rows 5-6 of 8

--- Extrayendo features de batch de 2 reseñas ---
MODEL_ID usado: meta-llama/llama-4-scout-17b-16e-instruct


...

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"


[WARN] intento 1/2 fallido en step2: Error code: 400 - {'error': {'message': "Generated JSON does not match the expected schema. Please adjust your prompt. See 'failed_generation' for more details. Error: jsonschema: '/results/0/aspects' does not validate with /properties/results/items/$ref/properties/aspects/maxItems: maximum 5 items required, but found 7 items", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': '{\n"results": [\n    {\n      "id": "R1",\n      "sentiment": "mixed",\n      "sentiment_score": 5,\n      "aspects": [\n        "gameplay",\n        "story",\n        "graphics",\n        "soundtrack",\n        "photo mode",\n        "performance",\n        "bugs"\n      ],\n      "feedback_type": "mixed"\n    },\n    {\n      "id": "R2",\n      "sentiment": "mixed",\n      "sentiment_score": 7,\n      "aspects": [\n        "gameplay",\n        "open-world",\n        "story",\n        "characters",\n        "soundtrack",\n        "bugs"\n 

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 0.5 segundos
Primeros 200 caracteres de la respuesta:
{
"results": [
    {
      "id": "R1",
      "sentiment": "mixed",
      "sentiment_score": 5,
      "aspects": [
        "gameplay",
        "graphics",
        "story",
        "bugs",
        "phot
Batch procesado



,Contenido,Valoración,Recomendado_binario,longitud_caracteres,relevant,reason,sentiment,sentiment_score,aspects,feedback_type
1,this was probably my first preorder i felt tha...,Recomendado,1,7662,True,"Concrete evaluation of visuals, gameplay, stor...",positive,9,"[city design, gameplay freedom, story, charact...",praise
3,i know many will handwave away any criticisms ...,No recomendado,0,7609,True,"Concrete evaluation of game mechanics, visuals...",negative,2,"[gameplay difficulty, enemy design, reward sys...",complaint
4,i had enormous expectations having put thousan...,No recomendado,0,7609,True,"Specific judgments on game mechanics, story, a...",negative,2,"[game design, story, combat, multiplayer, pres...",complaint
5,i finished the witcher 3 for the first time th...,Recomendado,1,7604,True,"Concrete evaluation of story, characters, worl...",mixed,6,"[story, characters, gameplay mechanics, bugs, ...",mixed
6,in any industry this will always happen as an ...,No recomendado,0,7590,True,"Specific judgments on AI, NPCs, customization,...",negative,2,"[ai, gameplay, character customization, story,...",complaint
7,now my favorite fromsoft game tldr here phenom...,Recomendado,1,7588,True,"Specific judgments on combat, world, story, pe...",positive,9,"[combat, world design, story, gameplay mechanics]",praise
8,i know im quite late to the review party and e...,Recomendado,1,7587,True,"Specific judgments on visuals, NPC AI, bugs, a...",mixed,5,"[gameplay, graphics, story, bugs, photo mode]",mixed
9,28 enero cyberpunk 2077 labor of love or broke...,Recomendado,1,7583,True,"Specific judgments on gameplay, visuals, story...",mixed,5,"[gameplay, open-world, story, bugs, rpg elements]",mixed


#### Afinamiento del prompt 

- **Sentiment y sentiment_score** 
    
    eviews muy negativas en tono:crashes, corrupted save files, griefing, QA problems se valoran como mixed porque se mencionana tanto aspecto positivos como negativos, sin embargo un humano probablemente lo marcaría como negativo.

    *If the review mainly criticizes the game despite mentioning some positives, classify it as negative.*

- **Aspects** 
    
    El modelo mezcla game features con entities

    *Aspects should describe general game features (e.g., gameplay, graphics, AI, story).*
    *Avoid specific proper nouns like locations or character names.*

##  Función para ejecutar por bloques y guardar progreso

Una vez iteramos y obtenemos resultados consistentes, ejecutamos sobre el dataframe completo. Para ello, desarrollamos una función que extrae subconjuntos del dataframe, los procesa y los guarda progresivamente (para protegernos ante posibles fallos de la API). De esta forma, si el proceso falla o se detiene, podemos reanudar la ejecución desde el último punto guardado sin perder el avance.


✔ no perdemos progreso.     
✔ podemos parar el notebook.   
✔ reducimos riesgo con APIs.   
✔ podemos inspeccionar resultados intermedios.   



In [ ]:
os.makedirs("data/results", exist_ok=True)
def run_with_checkpoint(df, chunk_size=10, text_col="Contenido", batch_size=2, sleep_s=30, output_path="data/results/reviews_with_features.pkl", continue_from_checkpoint=True):
    parts = []
    start = 0

    if continue_from_checkpoint and os.path.exists(output_path):
        saved = pd.read_pickle(output_path)
        parts.append(saved)
        start = len(saved)
        print(f"Reanudando desde fila {start}")

    elif not continue_from_checkpoint and os.path.exists(output_path):
        print("Empezando desde 0 (archivo existente será sobrescrito)")

    for i in range(start, len(df), chunk_size):

        chunk = df.iloc[i:i+chunk_size].copy()

        processed = run_step2_feature_extraction_groq(
            chunk,
            text_col=text_col,
            batch_size=batch_size,
            sleep_s=sleep_s
        )

        parts.append(processed)

        current_df = pd.concat(parts, ignore_index=True)
        current_df.to_pickle(output_path)

        print(f"Checkpoint guardado: {len(current_df)}/{len(df)} filas")

    return pd.concat(parts, ignore_index=True)

In [ ]:
# Coge chunks de 10 reviews,genera prompts con batches de 2 reviews y guarda checkpointer
df_results = run_with_checkpoint(df_to_evaluate, chunk_size=10, batch_size=2, sleep_s=30, output_path="data/results/results_step2.pkl", continue_from_checkpoint=False) 

Empezando desde 0 (archivo existente será sobrescrito)
[Batch 1/5] rows 1-2 of 10

--- Extrayendo features de batch de 2 reseñas ---
MODEL_ID usado: meta-llama/llama-4-scout-17b-16e-instruct


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 0.7 segundos
Batch procesado

[Batch 2/5] rows 3-4 of 10

--- Extrayendo features de batch de 2 reseñas ---
MODEL_ID usado: meta-llama/llama-4-scout-17b-16e-instruct


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 0.6 segundos
Batch procesado

[Batch 3/5] rows 5-6 of 10

--- Extrayendo features de batch de 2 reseñas ---
MODEL_ID usado: meta-llama/llama-4-scout-17b-16e-instruct


...

Groq respondió en 0.5 segundos
Batch procesado

Checkpoint guardado: 90/93 filas
[Batch 1/2] rows 1-2 of 3

--- Extrayendo features de batch de 2 reseñas ---
MODEL_ID usado: meta-llama/llama-4-scout-17b-16e-instruct


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 0.5 segundos
Batch procesado

[Batch 2/2] rows 3-3 of 3

--- Extrayendo features de batch de 1 reseñas ---
MODEL_ID usado: meta-llama/llama-4-scout-17b-16e-instruct


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Groq respondió en 0.4 segundos
Batch procesado

Checkpoint guardado: 93/93 filas


## Analizamos resultados

### 1. Evaluación general de resultados
Los resultados muestran que el modelo es capaz de extraer de forma consistente información estructurada a partir de reseñas largas y no estructuradas. En la mayoría de los casos, las categorías de sentimiento, puntuación y tipo de feedback reflejan adecuadamente el tono general de la reseña.

### 2. Sobre los aspects
El modelo identifica correctamente los temas (aspects) principales mencionados en las reseñas, aunque presentan cierta variabilidad semántica, esto podría mitigarse mediante una etapa adicional de normalización o clustering.

### 3. Sobre el sentiment
En algunos casos el sentimiento se clasifica como “mixed” cuando la reseña contiene críticas predominantes junto con algunos comentarios positivos. Este comportamiento es esperable, ya que el modelo tiende a capturar la ambivalencia presente en el texto en lugar de forzar una polaridad estricta.

### 4. Conclusiones

En conjunto, el pipeline desarrollado permite transformar reseñas extensas y no estructuradas en datos estructurados que pueden utilizarse posteriormente para análisis exploratorio o generación de insights. Para ello, se ha puesto especial énfasis en el uso de salidas estructuradas (structured outputs) y validación mediante esquemas (schema validation), lo que permite garantizar la consistencia del formato de salida.

Durante el desarrollo se han evaluado diferentes motores de inferencia y métodos de acceso a modelos, incluyendo ejecución local mediante **Ollama**, modelos cargados a través de **Hugging Face y APIs externas**. Asimismo, se han probado distintos modelos dentro de cada enfoque con el objetivo de comparar su comportamiento en tareas de extracción estructurada de información.

In [100]:
pd.set_option('display.max_colwidth', 200)
df_results.sample(6)

,Contenido,Valoración,Recomendado_binario,longitud_caracteres,relevant,reason,sentiment,sentiment_score,aspects,feedback_type
45,good looking mess i love cyberpunk fiction the original blade runner is one of my favorite movies of all time and ive been always fan of the theming when i heard of this back in 2013 it looked lik...,No recomendado,0,6398,True,"Specific judgments on visuals, character customization, progression system, and bugs",mixed,5,"[aesthetic, character creator, progression system, bugs, open world]",mixed
37,i think after 249 hours and three full runs through the game and seeing the wide array of potential endings i can safely say i feel i certainly got my moneys worth with cyberpunk 2077 i bet you ju...,No recomendado,0,6820,True,"Specific judgments on game mechanics, visuals, and potential",mixed,5,"[gameplay, world design, story, bugs, performance]",mixed
49,for the first time i am doing revised review for game originally i had given this game negative review because i felt the cons outweighed the pros but after patch 103 and continuing to play the ga...,Recomendado,1,6340,True,"Concrete evaluation of various game aspects, including graphics, gameplay, NPCs, and multiplayer",positive,8,"[graphics, gameplay, open world, npcs and quests, multiplayer]",mixed
78,this is basically negative review for people who like soulslikes and all previous souls and sekiro games if its your first game it might be fine for you albeit very hard on the main bosses ive pla...,No recomendado,0,5383,True,"Specific judgments on game mechanics, design, and combat system",negative,2,"[gameplay, open world, combat, story, performance]",complaint
59,as many other people have pointed out there are lot of bugs at this stage of the game this is to be expected the game is really huge in scope and theres bound to be bugs in release software is nev...,No recomendado,0,6105,True,"Specific judgments on bugs, visuals, story pacing, and game features",negative,4,"[bugs, customization, story, gameplay, performance]",complaint
91,dios hecho juego debería existirjugabilidad complicadoeste único juego conozco juegas primera vez haberlo jugado vez saber jugarlo juego tan difícil principio necesitarías haberlo jugado vez saber...,Recomendado,1,5167,True,"Specific judgments on gameplay, graphics, story, and mechanics",mixed,5,"[gameplay, difficulty, story, mechanics, challenge]",mixed


## Guardamos resultados

In [ ]:
# Realmente ya se guardan los checkpoints en cada iteración, pero mantenemos estas líneas comentadas 
# os.makedirs("data/results", exist_ok=True)
# df_results.to_pickle("data/results/reviews_with_features.pkl")

## Ejecución con Hugging Face que no pudimos llegar a ejecutar por limitaciones de hardware:

Para obtener robustez real en la respuesta del modelo probamos `LM Format Enforcer` (también existe `Outlines`). 
En Ollama, para el filtrado de reviews relevantes, hemos relaizado misma operativa con el argumento format=<json_schema>, con el objetivo de no depender exclusivamente del prompt para estructura de salida del mensaje.
En Hugging Face se hace distinto: transformers.generate() no tiene format=<json_schema> nativo como Ollama.

Diferencia clave:

-  **prompt + parse + fallback:**   
  El modelo genera libremente.Si sale mal, te enteras después (error).

- **constrained decoding (Outlines / LM Format Enforcer / Jsonformer):** 
    El modelo solo puede generar tokens válidos para el schema.
    Menos invalid_schema, menos retries/fallbacks.

Con estas herramientas le pasamos el schema/Pydantic y ellas fuerzan formato.   
En la práctica, una instrucción mínima en prompt sigue ayudando a calidad semántica (relevant correcto), aunque el formato ya vaya forzado.  


In [ ]:
SYSTEM_PROMPT_STEP2 = """You are a videogame review analyst for a product development team.
Extract structured insights from each review.
Return STRICT JSON only with one root object: {"results":[...]}.
No markdown, no extra text, no extra keys."""


USER_TEMPLATE_STEP2 = Template("""Extract structured information from each videogame review.

Output schema (JSON only):
{
  "results": [
    {
      "id": "R1",
      "sentiment": "positive | negative | mixed | neutral",
      "sentiment_score": 0,
      "aspects": ["aspect1", "aspect2"],
      "feedback_type": "praise | complaint | mixed"
    }
  ]
}

Definitions:
- sentiment: overall tone of the review.
- sentiment_score: opinion intensity from 0 to 10.
  - 0-3: mostly negative
  - 4-6: mixed or neutral
  - 7-10: mostly positive
- aspects: concrete game aspects explicitly mentioned (max 5 terms).
- feedback_type:
  - praise: mostly positive feedback
  - complaint: mostly negative feedback
  - mixed: both praise and criticism

Rules:
- Reviews can be in English or Spanish; apply the same criteria.
- Return exactly one result per input review.
- Preserve IDs exactly as provided (R1, R2, ...).
- Do not invent information not present in the review.
- Include only clearly mentioned aspects.
- Use at most 5 aspects.
- Output ONLY one JSON object with key "results".

Few-shot example:

Input:
<review id="R1">
The gameplay is fun and visuals are great, but the game crashes too often.
</review>

Output:
{
  "results": [
    {
      "id": "R1",
      "sentiment": "mixed",
      "sentiment_score": 5,
      "aspects": ["gameplay", "graphics", "stability"],
      "feedback_type": "mixed"
    }
  ]
}

REVIEWS:
$text
""")


# Consrucción de mensajes para el modelo, con formato específico para chat instructivo
def build_reviews_block_step2(reviews: list[str]) -> str:
    lines = []
    for i, r in enumerate(reviews, 1):
        rid = f"R{i}"
        r = (r or "").strip()
        lines.append(f'<review id="{rid}">\n{r}\n</review>')
    return "\n\n".join(lines)

def build_messages_for_batch_step2(reviews: list[str]) -> list[dict]:
    reviews_block = build_reviews_block_step2(reviews)
    user_prompt = USER_TEMPLATE_STEP2.substitute(text=reviews_block)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_STEP2},
        {"role": "user", "content": user_prompt},
    ]
    return messages


class ReviewFeatures(BaseModel):
    id: str
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    sentiment_score: int = Field(ge=0, le=10)
    aspects: conlist(str, max_length=5)
    feedback_type: Literal["praise", "complaint", "mixed"]

class BatchOutput(BaseModel):
    results: List[ReviewFeatures]

schema = BatchOutput.model_json_schema()   # Pydantic v2
parser = JsonSchemaParser(schema)
prefix_fn = build_transformers_prefix_allowed_tokens_fn(tokenizer, parser)



## Generación de respuesta 
# no estamos haciendo backpropagation, así que ahorramos memoria y aceleramos la inferencia con torch.no_grad()
@torch.no_grad() 
def hf_generate_from_messages(messages: list[dict], max_new_tokens: int = MAX_RESPONSE_TOKENS_FEAT) -> str:
    """
    Genera la respuesta del modelo a partir de mensajes chat.
    Usa apply_chat_template para respetar el formato esperado por modelos instruct/chat.
    """
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True, 
        max_length=MAX_INPUT_TOKENS_FEAT 
    ).to(device)

    print("Longitud prompt (chars):", len(prompt_text))
    print("Tokens de entrada:", inputs["input_ids"].shape[1])

    start = time.time()

    print("Total de tokens (entrada + salida):", inputs["input_ids"].shape[1] + MAX_RESPONSE_TOKENS_FEAT)


    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        prefix_allowed_tokens_fn=prefix_fn
    )

    elapsed = time.time() - start
    print(f"Modelo respondió en {elapsed:.1f} segundos")

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print("Primeros 200 caracteres de la respuesta:")
    print(response_text[:200])

    return response_text


# JSON parse
def parse_json_robust_step2(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        if "{" in text and "}" in text:
            cut = text[text.find("{"): text.rfind("}") + 1]
            return json.loads(cut)
        raise


# Extraccion de features
def extract_features_batch_hf(reviews: list[str]) -> list[dict]:
    print(f"\n--- Extrayendo features de batch de {len(reviews)} reseñas ---")
    messages = build_messages_for_batch_step2(reviews)
    try:
        raw = hf_generate_from_messages(messages)
        data = parse_json_robust_step2(raw)
        validated = BatchOutput(**data)
        results = validated.results

        if len(results) != len(reviews):
            raise ValueError(f"batch_mismatch: expected={len(reviews)} got={len(results)}")

        out = []
        for i, r in enumerate(results, start=1):
            out.append({
                "id": r.id if r.id else f"R{i}",
                "sentiment": r.sentiment,
                "sentiment_score": r.sentiment_score,
                "aspects": [a.strip().lower() for a in r.aspects][:5],
                "feedback_type": r.feedback_type,
            })

        print("Batch procesado\n")
        return out
    
    except Exception as e:
        print(f"[WARN] fallo batch step2: {e}", flush=True)
        return [{
            "id": f"R{i+1}",
            "sentiment": "neutral",
            "sentiment_score": 5,
            "aspects": [],
            "feedback_type": "mixed"
        } for i in range(len(reviews))]

# Aplicamos funciones 
def run_step2_feature_extraction_hf( df: pd.DataFrame, text_col: str = "Contenido", batch_size: int = 1, sleep_s: float = 0.0) -> pd.DataFrame:
    df = df.copy()
    texts = df[text_col].fillna("").astype(str).tolist()
    all_preds = []

    total = len(texts)
    total_batches = (total + batch_size - 1) // batch_size

    for batch_idx, start in enumerate(range(0, total, batch_size), start=1):
        end = min(start + batch_size, total)
        batch = texts[start:end]

        print(f"[Batch {batch_idx}/{total_batches}] rows {start+1}-{end} of {total}", flush=True)
        preds = extract_features_batch_hf(batch)

        if len(preds) != len(batch):
            preds = [{
                "id": f"R{i+1}",
                "sentiment": "neutral",
                "sentiment_score": 5,
                "aspects": [],
                "feedback_type": "mixed"
            } for i in range(len(batch))]

        all_preds.extend(preds)

        if sleep_s > 0:
            time.sleep(sleep_s)

    df["sentiment"] = [p["sentiment"] for p in all_preds]
    df["sentiment_score"] = [p["sentiment_score"] for p in all_preds]
    df["aspects"] = [p["aspects"] for p in all_preds]
    df["feedback_type"] = [p["feedback_type"] for p in all_preds]

    return df

In [ ]:
df_features_test = run_step2_feature_extraction_hf(df_to_evaluate[:1], text_col="Contenido", batch_size=1,sleep_s=0)


--- Extrayendo features de batch de 1 reseñas ---
Longitud prompt (chars): 9379
Tokens de entrada: 1962
input_tokens: 1962
total_budget: 2462


/opt/miniconda3/envs/gf311_cf_entrega/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/miniconda3/envs/gf311_cf_entrega/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/miniconda3/envs/gf311_cf_entrega/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


KeyboardInterrupt: 